# Equações Exponenciais e Logarítmicas

**Módulo:** 02 – Logaritmos

**Pré-requisitos:** [`0204a_funcao_logaritmica.ipynb`](0204a_funcao_logaritmica.ipynb) (logaritmo como inversa da exponencial), [`0203a_logaritmos.ipynb`](0203a_logaritmos.ipynb) (propriedades operatórias e condições de existência) e [`0202a_funcao_exponencial.ipynb`](0202a_funcao_exponencial.ipynb) (equações exponenciais simples, de base igual)

**Objetivos de aprendizagem:**
- Resolver equações exponenciais gerais (não redutíveis a uma base comum) usando logaritmo, incluindo casos por substituição de variável.
- Resolver equações logarítmicas diretas e equações com logaritmo dos dois lados, verificando sempre as condições de existência.
- Reconhecer e resolver equações mistas, que combinam termos exponenciais e logarítmicos.
- Resolver numérica e graficamente, com Python, equações que não têm solução algébrica simples.

**Tempo estimado:** 80 a 100 minutos

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/02_logaritmos/0205a_equacoes_exponenciais_e_logaritmicas.ipynb)

## Quantos anos tem esse fóssil?

Um fóssil foi encontrado com apenas 30% do carbono-14 original ainda presente em sua composição. O carbono-14 é um isótopo radioativo que decai com o tempo, e sua meia-vida — o tempo que leva para metade da quantidade inicial desaparecer — é de aproximadamente 5.730 anos.

**Quantos anos tem esse fóssil?**

A equação por trás dessa pergunta é exponencial: se N₀ é a quantidade inicial de carbono-14 e N(t) é a quantidade restante depois de t anos,

$$N(t) = N_0 \cdot \left(\frac{1}{2}\right)^{t/5730}$$

e queremos descobrir o t que faz N(t)/N₀ = 0,30. Só que essa não é uma daquelas equações "de sorte", em que os dois lados viram potências da mesma base (como em `0202a_funcao_exponencial`) — aqui, 0,30 não é uma potência exata de 1/2.

Ao final deste notebook, você vai resolver esse problema com precisão — e vai sair daqui com a técnica geral para resolver **qualquer** equação exponencial ou logarítmica, não só as de "base igual".

📖 **Leitura complementar:** [Datação por radiocarbono](https://pt.wikipedia.org/wiki/Data%C3%A7%C3%A3o_por_radiocarbono)

In [2]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import brentq

# Paleta de cores Nord
NORD_FUNDO = "#2E3440"
NORD_PAINEL = "#3B4252"
NORD_LINHA = "#4C566A"
NORD_TEXTO = "#ECEFF4"
NORD_TEXTO_SEC = "#D8DEE9"
COR_PONTO = "#88C0D0"
COR_DESTAQUE = "#A3BE8C"
COR_ALERTA = "#BF616A"
COR_SECUNDARIA = "#81A1C1"
COR_EXTRA = "#B48EAD"
COR_AVISO = "#EBCB8B"


def estilizar_eixo(ax) -> None:
    '''Aplica o estilo Nord padrão a um eixo do matplotlib.'''
    ax.set_facecolor(NORD_FUNDO)
    ax.tick_params(colors=NORD_TEXTO_SEC)
    for spine in ax.spines.values():
        spine.set_color(NORD_LINHA)
    ax.grid(color=NORD_LINHA, linewidth=0.5, alpha=0.5)

## 1. Equações Exponenciais Gerais (Usando Logaritmo)

📌 **Definição formal:** para resolver uma equação exponencial que não se reduz a uma base comum, isolamos a potência de um lado e aplicamos logaritmo dos dois lados. A propriedade log(aˣ) = x·log(a) "desce" o expoente e transforma a equação numa equação linear em x:

$$a^x = b \implies \log(a^x) = \log(b) \implies x \cdot \log(a) = \log(b) \implies x = \frac{\log(b)}{\log(a)}$$

Repare que essa fórmula é exatamente a definição de logaritmo com mudança de base: x = log_a(b), como já vimos em `0203a_logaritmos`. A novidade aqui não é o resultado, é o **caminho** — sabemos chegar em x mesmo sem lembrar a fórmula de cor, só aplicando log nos dois lados.

In [2]:
# Revisão rápida: 2^x = 10, já resolvido em 0204a_funcao_logaritmica
x = math.log(10) / math.log(2)
print("2^x = 10")
print("log(2^x) = log(10)  ->  x*log(2) = log(10)")
print(f"x = log(10)/log(2) = {x:.4f}")

# Agora um caso com coeficiente: em quantos anos um investimento TRIPLICA
# de valor, a uma taxa de juros compostos de 8% ao ano?
# C*(1.08)^t = 3*C  ->  (1.08)^t = 3  (o capital inicial C se cancela)
taxa = 0.08
t = math.log(3) / math.log(1 + taxa)
print(f"\n(1.08)^t = 3")
print(f"t*log(1.08) = log(3)  ->  t = log(3)/log(1.08) = {t:.4f} anos")
print(f"conferindo: 1.08^{t:.4f} = {(1 + taxa)**t:.4f}  (bate com 3)")

2^x = 10
log(2^x) = log(10)  ->  x*log(2) = log(10)
x = log(10)/log(2) = 3.3219

(1.08)^t = 3
t*log(1.08) = log(3)  ->  t = log(3)/log(1.08) = 14.2749 anos
conferindo: 1.08^14.2749 = 3.0000  (bate com 3)


**Pergunta de checagem:** por que aplicar logaritmo dos dois lados de uma equação não muda a igualdade?

🎯 **Aplicação prática:** essa é a conta que qualquer calculadora financeira faz por trás dos panos para responder "em quanto tempo meu dinheiro dobra/triplica a uma taxa x?".

## 2. Equações Exponenciais por Substituição de Variável

📌 **Definição formal:** quando a equação envolve potências de aˣ elevadas a outros expoentes (como a^(2x), que é (aˣ)²), fazemos a substituição y = aˣ. Isso transforma a equação exponencial numa equação polinomial (em geral, quadrática) em y. Resolvemos para y e, para **cada valor positivo** de y, resolvemos aˣ = y separadamente — descartando qualquer y ≤ 0, porque aˣ é sempre positivo, não importa o valor de x.

In [3]:
# Resolvendo 4^x - 5*2^x + 4 = 0 por substituição: y = 2^x
# Repare que 4^x = (2^2)^x = (2^x)^2 = y^2
y = sp.symbols("y")
equacao_em_y = sp.Eq(y**2 - 5 * y + 4, 0)
raizes_y = sp.solve(equacao_em_y, y)
print("equação original: 4^x - 5*2^x + 4 = 0")
print("substituindo y = 2^x:", equacao_em_y, " ->  raízes em y:", raizes_y)

print("\nvoltando para x (resolvendo 2^x = y para cada raiz positiva):")
for valor_y in raizes_y:
    if valor_y <= 0:
        print(f"  y = {valor_y}  ->  descartado (2^x nunca é <= 0)")
        continue
    valor_x = sp.log(valor_y, 2)
    print(f"  y = {valor_y}  ->  2^x = {valor_y}  ->  x = {valor_x}")

equação original: 4^x - 5*2^x + 4 = 0
substituindo y = 2^x: Eq(y**2 - 5*y + 4, 0)  ->  raízes em y: [1, 4]

voltando para x (resolvendo 2^x = y para cada raiz positiva):
  y = 1  ->  2^x = 1  ->  x = 0
  y = 4  ->  2^x = 4  ->  x = 2


🕹️ **Widget interativo:** ajuste a base a e os coeficientes p, q da equação (aˣ)² + p·aˣ + q = 0. O widget mostra a equação polinomial em y, resolve para y e converte de volta para x, descartando as raízes inválidas.

In [4]:
def substituicao_exponencial(a: float, p: float, q: float) -> None:
    y = sp.symbols("y")
    raizes_y = sp.solve(sp.Eq(y**2 + p * y + q, 0), y)
    print(f"equação: ({a:g}^x)^2 + ({p:g})*{a:g}^x + ({q:g}) = 0")
    print(f"substituindo y = {a:g}^x:  y^2 + ({p:g})y + ({q:g}) = 0  ->  y = {raizes_y}")
    print()
    encontrou_solucao = False
    for valor_y in raizes_y:
        if (not valor_y.is_real) or valor_y <= 0:
            print(f"  y = {valor_y}  ->  descartado ({a:g}^x nunca é <= 0 ou não é real)")
            continue
        valor_x = sp.log(valor_y, a)
        encontrou_solucao = True
        print(f"  y = {valor_y}  ->  {a:g}^x = {valor_y}  ->  x = {sp.nsimplify(valor_x)} ~= {float(valor_x):.4f}")
    if not encontrou_solucao:
        print("\nnenhuma raiz válida -- a equação não tem solução real.")


widgets.interact(
    substituicao_exponencial,
    a=widgets.FloatSlider(value=2.0, min=1.1, max=5.0, step=0.1, description="a:"),
    p=widgets.FloatSlider(value=-5.0, min=-10.0, max=10.0, step=0.5, description="p:"),
    q=widgets.FloatSlider(value=4.0, min=-10.0, max=10.0, step=0.5, description="q:"),
)

interactive(children=(FloatSlider(value=2.0, description='a:', max=5.0, min=1.1), FloatSlider(value=-5.0, desc…

<function __main__.substituicao_exponencial(a: float, p: float, q: float) -> None>

**Pergunta de checagem:** por que, ao encontrar os valores de y, é preciso descartar os negativos (ou o zero)?

🎯 **Aplicação prática:** esse tipo de equação aparece em modelos que combinam dois termos exponenciais da mesma base — como a carga e a descarga de um capacitor em circuitos elétricos RC.

📖 **Leitura complementar:** [Função exponencial](https://pt.wikipedia.org/wiki/Fun%C3%A7%C3%A3o_exponencial)

## 3. Equações Exponenciais com Bases Diferentes

📌 **Definição formal:** quando as bases não são iguais nem relacionáveis por uma potência comum (como 2ˣ = 3^(x+1), já que 2 e 3 não têm base comum), aplicamos logaritmo dos dois lados e usamos as propriedades operatórias para colocar x em evidência e isolá-lo algebricamente.

$$2^x = 3^{x+1} \implies \log(2^x) = \log(3^{x+1}) \implies x\log 2 = (x+1)\log 3$$

Como x aparece nos dois lados, agrupamos os termos com x:

$$x\log 2 - x\log 3 = \log 3 \implies x(\log 2 - \log 3) = \log 3 \implies x = \frac{\log 3}{\log 2 - \log 3}$$

In [5]:
# Resolvendo 2^x = 3^(x+1) passo a passo
log2, log3 = math.log(2), math.log(3)
x = log3 / (log2 - log3)
print("2^x = 3^(x+1)")
print("x*log(2) = (x+1)*log(3)")
print("x*log(2) - x*log(3) = log(3)")
print("x*(log(2) - log(3)) = log(3)")
print(f"x = log(3) / (log(2) - log(3)) = {x:.4f}")
print(f"conferindo: 2^{x:.4f} = {2**x:.4f}   e   3^{x + 1:.4f} = {3**(x + 1):.4f}")

# Aplicação: dois investimentos com capital inicial e taxa diferentes -- quando se igualam?
# investimento A: R$1000 a 10% ao ano | investimento B: R$1500 a 6% ao ano
capital_a, taxa_a = 1000, 0.10
capital_b, taxa_b = 1500, 0.06

# capital_a*(1+taxa_a)^t = capital_b*(1+taxa_b)^t
# log(capital_a) + t*log(1+taxa_a) = log(capital_b) + t*log(1+taxa_b)
t_igualdade = (math.log(capital_b) - math.log(capital_a)) / (
    math.log(1 + taxa_a) - math.log(1 + taxa_b)
)
print(f"\ninvestimento A (R${capital_a}, {taxa_a:.0%} a.a.) alcança investimento B (R${capital_b}, {taxa_b:.0%} a.a.)")
print(f"em t = {t_igualdade:.2f} anos")
print(f"conferindo: A = {capital_a * (1 + taxa_a)**t_igualdade:.2f}   B = {capital_b * (1 + taxa_b)**t_igualdade:.2f}")

2^x = 3^(x+1)
x*log(2) = (x+1)*log(3)
x*log(2) - x*log(3) = log(3)
x*(log(2) - log(3)) = log(3)
x = log(3) / (log(2) - log(3)) = -2.7095
conferindo: 2^-2.7095 = 0.1529   e   3^-1.7095 = 0.1529

investimento A (R$1000, 10% a.a.) alcança investimento B (R$1500, 6% a.a.)
em t = 10.95 anos
conferindo: A = 2838.55   B = 2838.55


**Pergunta de checagem:** depois de aplicar log nos dois lados, por que x acaba aparecendo em mais de um termo — e como resolvemos isso?

🎯 **Aplicação prática:** essa técnica resolve, por exemplo, "em que momento dois investimentos com taxas e valores iniciais diferentes se igualam?" (veja o exemplo acima).

## 4. Equações Logarítmicas Diretas

📌 **Definição formal:** para resolver uma equação em que o logaritmo já está isolado de um lado, "desfazemos" o logaritmo aplicando a definição — exponenciando os dois lados na base do logaritmo:

$$\log_a(f(x)) = c \iff f(x) = a^c$$

Como sempre com logaritmo, toda solução encontrada precisa satisfazer f(x) > 0 (mais sobre isso no bloco 6).

In [6]:
# a) log_2(x) = 5  ->  x = 2^5
x_a = 2**5
print(f"a) log_2(x) = 5        ->  x = 2^5 = {x_a}")

# b) log_3(2x - 1) = 2  ->  2x - 1 = 3^2  ->  x = (9+1)/2
x_b = (3**2 + 1) / 2
print(f"b) log_3(2x-1) = 2     ->  2x-1 = 3^2 = 9   ->  x = {x_b}")
print(f"   conferindo: log_3({2 * x_b - 1:g}) = {math.log(2 * x_b - 1, 3):.4f}")

# c) log_5(x+3) = 1  ->  x+3 = 5^1  ->  x = 2
x_c = 5**1 - 3
print(f"c) log_5(x+3) = 1      ->  x+3 = 5           ->  x = {x_c}")

# Aplicação: escala Richter -- M = M0 + log10(razão de amplitude)
# um terremoto de referência tem magnitude 4.5; outro foi 50x mais forte em amplitude
magnitude_referencia = 4.5
razao = 50
magnitude_novo = magnitude_referencia + math.log10(razao)
print(f"\nRichter: M = {magnitude_referencia} + log10({razao}) = {magnitude_novo:.2f}")

a) log_2(x) = 5        ->  x = 2^5 = 32
b) log_3(2x-1) = 2     ->  2x-1 = 3^2 = 9   ->  x = 5.0
   conferindo: log_3(9) = 2.0000
c) log_5(x+3) = 1      ->  x+3 = 5           ->  x = 2

Richter: M = 4.5 + log10(50) = 6.20


**Pergunta de checagem:** qual operação "desfaz" um logaritmo, e por que ela funciona?

🎯 **Aplicação prática:** calcular a magnitude de um terremoto sabendo quantas vezes mais forte ele foi que outro de magnitude conhecida — retomando a escala Richter de `0204a_funcao_logaritmica`.

## 5. Equações Logarítmicas com Logaritmo dos Dois Lados

📌 **Definição formal:** quando ambos os lados da equação são logaritmos de mesma base, podemos "cancelar" os logaritmos porque a função logarítmica é **injetora** (biunívoca) — cada saída corresponde a uma única entrada:

$$\log_a(f(x)) = \log_a(g(x)) \implies f(x) = g(x)$$

In [7]:
# log(x^2 + 5) = log(6x)  ->  x^2 + 5 = 6x  (mesma base: log decimal nos dois lados)
x = sp.symbols("x", real=True)
equacao = sp.Eq(x**2 + 5, 6 * x)
raizes = sp.solve(equacao, x)
print("log(x^2+5) = log(6x)")
print("x^2+5 = 6x  ->  x^2-6x+5 = 0")
print("raízes:", raizes)

print("\nconferindo as condições de existência:")
print("x^2+5 > 0 sempre vale (soma de quadrado com número positivo); falta checar 6x > 0:")
for r in raizes:
    situacao = "> 0, válido" if 6 * r > 0 else "<= 0, inválido"
    print(f"  x = {r}: 6x = {6 * r} {situacao}")

log(x^2+5) = log(6x)
x^2+5 = 6x  ->  x^2-6x+5 = 0
raízes: [1, 5]

conferindo as condições de existência:
x^2+5 > 0 sempre vale (soma de quadrado com número positivo); falta checar 6x > 0:
  x = 1: 6x = 6 > 0, válido
  x = 5: 6x = 30 > 0, válido


**Pergunta de checagem:** por que podemos "cancelar" os logaritmos dos dois lados quando eles têm exatamente a mesma base?

🎯 **Aplicação prática:** a mesma ideia (injetividade) garante que, se dois processos com crescimento logarítmico atingem o mesmo valor, foi porque partiram do mesmo ponto — útil em comparações científicas.

## 6. Verificação das Condições de Existência e Soluções Estranhas

⚠️ **Erro comum:** encontrar as soluções "matemáticas" de uma equação transformada e esquecer de testá-las nas condições de existência da equação **original** — todo logaritmando precisa ser positivo. Transformar a equação (elevar, expandir um produto de logs, etc.) pode introduzir soluções que resolvem a equação transformada, mas que geram log de número não positivo na equação de partida. Essas soluções precisam ser descartadas.

In [8]:
# log(x) + log(x - 3) = log(4)
# usando a propriedade do produto (vista em 0203a_logaritmos): log(x)+log(x-3) = log(x*(x-3))
x = sp.symbols("x", real=True)
equacao_transformada = sp.Eq(x * (x - 3), 4)
raizes = sp.solve(equacao_transformada, x)
print("log(x) + log(x-3) = log(4)")
print("log(x*(x-3)) = log(4)  ->  x*(x-3) = 4  ->  x^2 - 3x - 4 = 0")
print("raízes 'matemáticas':", raizes)

print("\ncondições de existência da equação ORIGINAL: x > 0  E  x - 3 > 0 (ou seja, x > 3)")
for r in raizes:
    valido = (r > 0) and (r - 3 > 0)
    print(f"  x = {r}: x>0? {r > 0}   x-3>0? {r - 3 > 0}   ->  {'VÁLIDO' if valido else 'DESCARTADO'}")

log(x) + log(x-3) = log(4)
log(x*(x-3)) = log(4)  ->  x*(x-3) = 4  ->  x^2 - 3x - 4 = 0
raízes 'matemáticas': [-1, 4]

condições de existência da equação ORIGINAL: x > 0  E  x - 3 > 0 (ou seja, x > 3)
  x = -1: x>0? False   x-3>0? False   ->  DESCARTADO
  x = 4: x>0? True   x-3>0? True   ->  VÁLIDO


🕹️ **Widget interativo:** informe um candidato a solução de log(x) + log(x-3) = log(4). O widget verifica automaticamente se ele satisfaz as condições de existência (e, se satisfizer, se de fato resolve a equação).

In [9]:
def verificar_candidato(x_candidato: float) -> None:
    print(f"candidato: x = {x_candidato:g}")
    condicao_1 = x_candidato > 0
    condicao_2 = (x_candidato - 3) > 0
    print(f"  log(x)   exige x > 0      ->  x = {x_candidato:g}      {'OK' if condicao_1 else 'FALHA'}")
    print(f"  log(x-3) exige x-3 > 0    ->  x-3 = {x_candidato - 3:g}   {'OK' if condicao_2 else 'FALHA'}")
    if condicao_1 and condicao_2:
        lado_esquerdo = math.log(x_candidato) + math.log(x_candidato - 3)
        lado_direito = math.log(4)
        satisfaz = abs(lado_esquerdo - lado_direito) < 1e-6
        print(f"  condições de existência OK -- testando a equação: {'satisfaz' if satisfaz else 'NÃO satisfaz'}")
        print("  => SOLUÇÃO VÁLIDA" if satisfaz else "  => passa no teste de existência, mas não resolve esta equação")
    else:
        print("  => SOLUÇÃO DESCARTADA (gera log de número não positivo)")


widgets.interact(
    verificar_candidato,
    x_candidato=widgets.FloatSlider(value=4.0, min=-2.0, max=6.0, step=0.5, description="x candidato:"),
)

interactive(children=(FloatSlider(value=4.0, description='x candidato:', max=6.0, min=-2.0, step=0.5), Output(…

<function __main__.verificar_candidato(x_candidato: float) -> None>

**Pergunta de checagem:** por que uma equação pode ter uma solução "matemática" (que resolve a equação transformada) mas não uma solução "válida" (para a equação original)?

🎯 **Aplicação prática:** evitar um dos erros mais comuns em provas e vestibulares — a verificação de solução em equações logarítmicas não é opcional, é parte do processo de resolução.

## 7. Equações Mistas (Exponencial e Logaritmo Juntos)

📌 **Definição formal:** algumas equações misturam uma base variável com um expoente logarítmico (ou vice-versa). Duas técnicas resolvem a maioria dos casos: usar a propriedade a^(log_a x) = x para simplificar diretamente, ou aplicar logaritmo dos dois lados e substituir y = log(x) para reduzir a uma única equação polinomial.

In [10]:
# Exemplo 1: usando a propriedade a^(log_a x) = x para simplificar diretamente
# 2^(log_2(x) + 3) = 40
# 2^(log_2(x)+3) = 2^(log_2 x) * 2^3 = x * 8   (aplicando a^(log_a x) = x)
x = sp.symbols("x", positive=True)
solucao = sp.solve(sp.Eq(x * 8, 40), x)
print("2^(log_2(x) + 3) = 40")
print("2^(log_2 x) * 2^3 = 40  ->  x * 8 = 40  ->  x =", solucao[0])
print(f"conferindo: 2^(log2({solucao[0]})+3) = {2**(math.log2(float(solucao[0])) + 3):.4f}")

# Exemplo 2: quando a propriedade não se aplica direto, aplicamos log e substituímos y = log(x)
# x^(log x) = 100  (logaritmo decimal)
y = sp.symbols("y", real=True)
raizes_y = sp.solve(sp.Eq(y**2, 2), y)
print("\nx^(log x) = 100")
print("log(x^(log x)) = log(100)  ->  (log x)^2 = 2")
print("substituindo y = log(x):  y^2 = 2  ->  y =", raizes_y)
print("voltando para x (y = log(x)  ->  x = 10^y):")
for valor_y in raizes_y:
    print(f"  y = {valor_y}  ->  x = 10^({valor_y}) ~= {float(10**valor_y):.4f}")

2^(log_2(x) + 3) = 40
2^(log_2 x) * 2^3 = 40  ->  x * 8 = 40  ->  x = 5
conferindo: 2^(log2(5)+3) = 40.0000

x^(log x) = 100
log(x^(log x)) = log(100)  ->  (log x)^2 = 2
substituindo y = log(x):  y^2 = 2  ->  y = [-sqrt(2), sqrt(2)]
voltando para x (y = log(x)  ->  x = 10^y):
  y = -sqrt(2)  ->  x = 10^(-sqrt(2)) ~= 0.0385
  y = sqrt(2)  ->  x = 10^(sqrt(2)) ~= 25.9546


**Pergunta de checagem:** qual propriedade permite simplificar diretamente uma expressão que mistura exponencial e logaritmo de mesma base?

🎯 **Aplicação prática:** situações em que um modelo de crescimento (exponencial) é medido numa escala logarítmica (como Richter ou decibéis) misturam os dois mundos — entender essa mistura ajuda a interpretar corretamente o resultado.

## 8. Resolução Numérica e Gráfica com Python

📌 **Conexão:** nem toda equação exponencial ou logarítmica tem uma solução algébrica simples de isolar. Quando a álgebra não é suficiente, podemos resolver **numericamente** (encontrando a raiz de uma função com `scipy.optimize`) ou **graficamente** (encontrando onde duas curvas se cruzam) — as duas abordagens são, na prática, a mesma ideia.

In [11]:
# Fechando o gancho: um fóssil tem 30% do carbono-14 original.
# N(t)/N0 = (1/2)^(t/5730) = 0.30
meia_vida = 5730  # anos
fracao_restante = 0.30

# Método algébrico (técnica do bloco 1): isolar a potência e aplicar log
idade_algebrica = meia_vida * math.log(fracao_restante) / math.log(0.5)
print(f"(1/2)^(t/{meia_vida}) = {fracao_restante}")
print(f"t = {meia_vida} * log({fracao_restante}) / log(0.5) = {idade_algebrica:.1f} anos")


# Método numérico: definir f(t) = (1/2)^(t/meia_vida) - fracao_restante e achar a raiz
def diferenca_fossil(t: float) -> float:
    return (0.5) ** (t / meia_vida) - fracao_restante


idade_numerica = brentq(diferenca_fossil, 0, 50_000)
print(f"\nconferindo com método numérico (scipy.optimize.brentq): t = {idade_numerica:.1f} anos")

resposta = f"o fóssil tem aproximadamente {idade_algebrica:,.0f} anos".replace(",", ".")
print(f"\nresposta: {resposta}.")

(1/2)^(t/5730) = 0.3
t = 5730 * log(0.3) / log(0.5) = 9952.8 anos

conferindo com método numérico (scipy.optimize.brentq): t = 9952.8 anos

resposta: o fóssil tem aproximadamente 9.953 anos.


🕹️ **Widget interativo:** ajuste os parâmetros de f(x) = aˣ e g(x) = mx + b. O widget plota as duas curvas, localiza numericamente onde elas se cruzam e destaca cada ponto de interseção — mesmo quando não existe fórmula algébrica fechada para x.

In [12]:
def cruzamento_exponencial_linear(a: float, m: float, b: float) -> None:
    def diferenca(x):
        return a**x - (m * x + b)

    xs = np.linspace(-5, 5, 800)
    ys_exp = a**xs
    ys_reta = m * xs + b

    raizes = []
    for x0, x1 in zip(xs[:-1], xs[1:]):
        if diferenca(x0) * diferenca(x1) < 0:
            raizes.append(brentq(diferenca, x0, x1))

    fig, ax = plt.subplots(figsize=(7, 5))
    fig.patch.set_facecolor(NORD_FUNDO)
    estilizar_eixo(ax)

    ax.plot(xs, ys_exp, color=COR_PONTO, linewidth=2.2, label=f"f(x) = {a:g}ˣ")
    ax.plot(xs, ys_reta, color=COR_SECUNDARIA, linewidth=2.2, label=f"g(x) = {m:g}x + {b:g}")
    for r in raizes:
        ax.plot(r, a**r, "o", color=COR_ALERTA, markersize=9, zorder=5)
        ax.annotate(
            f"({r:.2f}, {a**r:.2f})",
            (r, a**r),
            textcoords="offset points",
            xytext=(8, 8),
            color=NORD_TEXTO,
            fontsize=9,
        )

    limite_superior = min(max(ys_exp.max(), ys_reta.max()), 40)
    limite_inferior = min(ys_reta.min(), 0) - 2
    ax.set_ylim(limite_inferior, limite_superior)
    ax.set_xlabel("x", color=NORD_TEXTO_SEC)
    ax.set_ylabel("y", color=NORD_TEXTO_SEC)
    ax.legend(facecolor=NORD_PAINEL, labelcolor=NORD_TEXTO, edgecolor=NORD_LINHA, fontsize=10)
    titulo = f"{len(raizes)} interseção(ões) encontrada(s)" if raizes else "nenhuma interseção neste intervalo"
    ax.set_title(titulo, color=NORD_TEXTO, fontsize=10)
    plt.show()

    if raizes:
        print("soluções encontradas numericamente (sem fórmula algébrica fechada para elas):")
        for r in raizes:
            print(f"  x = {r:.4f}")
    else:
        print("nenhuma raiz visível neste intervalo.")


widgets.interact(
    cruzamento_exponencial_linear,
    a=widgets.FloatSlider(value=2.0, min=1.1, max=4.0, step=0.1, description="a:"),
    m=widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.5, description="m:"),
    b=widgets.FloatSlider(value=4.0, min=-5.0, max=10.0, step=0.5, description="b:"),
)

interactive(children=(FloatSlider(value=2.0, description='a:', max=4.0, min=1.1), FloatSlider(value=1.0, descr…

<function __main__.cruzamento_exponencial_linear(a: float, m: float, b: float) -> None>

**Pergunta de checagem:** mesmo com toda a álgebra aprendida neste notebook, por que ainda é útil saber resolver equações numericamente com Python?

🎯 **Aplicação prática:** é o próprio problema do gancho, finalmente resolvido (veja o código acima).

## Resumo — cheat sheet

| Conceito | Ideia-chave |
|---|---|
| Equação exponencial geral | aplicar log dos dois lados; usar log(aˣ) = x·log(a) para isolar x |
| Substituição y = aˣ | transforma numa equação polinomial; sempre descartar y ≤ 0 |
| Bases diferentes | log dos dois lados + agrupar os termos com x para colocá-lo em evidência |
| Equação log direta | log_a(f(x)) = c  ⟹  f(x) = aᶜ |
| Log = log (mesma base) | log_a(f(x)) = log_a(g(x))  ⟹  f(x) = g(x), pela injetividade do logaritmo |
| Verificação de existência | **sempre** obrigatória — descartar soluções que geram log de número não positivo |
| Equação mista | usar a^(log_a x) = x para simplificar, ou substituir y = log(x) |
| Sem solução algébrica simples | resolver numérica (`scipy.optimize.brentq`) ou graficamente (interseção de curvas) |

## Exercícios

**Básico**
1. Resolva uma equação exponencial geral usando logaritmo.
2. Resolva uma equação logarítmica direta, verificando a condição de existência.

**Intermediário**
3. Resolva uma equação exponencial por substituição de variável.
4. Resolva uma equação log = log (mesma base) que produz uma raiz a ser descartada por violar a condição de existência.

**Desafio**
5. Resolva um problema real (decaimento de um poluente) que exige montar e resolver uma equação exponencial geral, verificando o resultado.

Gabarito completo com resolução passo a passo em [`0205b_equacoes_exponenciais_e_logaritmicas_gabarito.ipynb`](0205b_equacoes_exponenciais_e_logaritmicas_gabarito.ipynb) — tente resolver sozinho antes de olhar.

### Básico

In [ ]:
# Exercício Básico 1: resolva a equação exponencial 5^x = 40 usando logaritmo.
# Calcule x com precisão decimal (não arredonde para inteiro).
x_exercicio1 = ...

print(f"{x_exercicio1:.4f}")

In [ ]:
# Verificação
x_esperado = math.log(40) / math.log(5)
assert abs(x_exercicio1 - x_esperado) < 1e-6, (
    "Tente de novo, revise: aplique log dos dois lados de 5^x=40 e isole x "
    "(seção 1 - Equações Exponenciais Gerais)."
)
print("Certo!")

In [ ]:
# Exercício Básico 2: resolva a equação logarítmica log_4(x) = 3.
# Em seguida, verifique se a solução satisfaz a condição de existência (x>0).
x_exercicio2 = ...
existe_exercicio2 = ...  # True ou False

print(x_exercicio2, existe_exercicio2)

In [ ]:
# Verificação
assert x_exercicio2 == 64, (
    "Tente de novo, revise: log_4(x)=3  ->  x = 4^3 (seção 4 - Equações Logarítmicas Diretas)."
)
assert existe_exercicio2 is True, (
    "Tente de novo, revise: x=64 é positivo, então satisfaz a condição de existência do log "
    "(seção 4 - Equações Logarítmicas Diretas)."
)
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 3: resolva 9^x - 4*3^x + 3 = 0 por substituição y = 3^x.
# Existem DUAS soluções reais para x. Guarde-as, em ordem crescente, numa lista.
solucoes_exercicio3 = ...  # lista com os dois valores de x, ex: [x1, x2]

print(solucoes_exercicio3)

In [ ]:
# Verificação
assert solucoes_exercicio3 == [0, 1], (
    "Tente de novo, revise: com y=3^x, a equação vira y^2-4y+3=0, cujas raízes são y=1 e y=3; "
    "y=1 -> x=0 e y=3 -> x=1 (seção 2 - Substituição de Variável)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 4: resolva log(x^2 - 1) = log(3x - 3).
# ATENÇÃO: uma das raízes algébricas deve ser descartada por violar
# uma condição de existência. Guarde só a solução VÁLIDA.
x_exercicio4 = ...

print(x_exercicio4)

In [ ]:
# Verificação
assert x_exercicio4 == 2, (
    "Tente de novo, revise: x^2-1=3x-3 dá x=1 ou x=2, mas em x=1 temos x^2-1=0, "
    "que não é > 0 -- só x=2 satisfaz as condições de existência "
    "(seção 6 - Verificação das Condições de Existência)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 5: a concentração de um poluente em um lago decai pela
# metade a cada 3 anos (decaimento exponencial). Hoje a concentração é de
# 80 mg/L, e a lei ambiental permite no máximo 5 mg/L. Calcule, com logaritmo
# (resposta exata, sem arredondar para inteiro), em quantos anos a
# concentração atinge o limite legal.
concentracao_inicial = 80
limite_legal = 5
meia_vida_poluente = 3  # anos

anos_ate_limite = ...

print(f"{anos_ate_limite:.4f} anos")

In [ ]:
# Verificação
anos_esperado = meia_vida_poluente * math.log(limite_legal / concentracao_inicial) / math.log(0.5)
assert abs(anos_ate_limite - anos_esperado) < 1e-6, (
    "Tente de novo, revise: concentracao_inicial*(1/2)^(t/meia_vida) = limite_legal "
    "-> isole a potência e aplique log (seção 1 - Equações Exponenciais Gerais)."
)
print("Certo!")


# Conferindo com o método numérico (seção 8), como sanity check
def diferenca_poluente(t: float) -> float:
    return concentracao_inicial * (0.5) ** (t / meia_vida_poluente) - limite_legal


anos_numerico = brentq(diferenca_poluente, 0, 200)
print(f"conferindo com método numérico: {anos_numerico:.4f} anos")

## Glossário do bloco

| Termo | Definição |
|---|---|
| Equação exponencial geral | equação com incógnita no expoente, não redutível a base comum, resolvida com logaritmo |
| Substituição de variável | trocar aˣ por y para transformar a equação exponencial numa equação polinomial |
| Equação logarítmica direta | equação com logaritmo isolado de um lado; resolvida exponenciando os dois lados |
| Injetividade do logaritmo | cada saída de log_a(x) corresponde a uma única entrada; permite cancelar log_a(f)=log_a(g) ⟹ f=g |
| Solução estranha (ou espúria) | solução algébrica da equação transformada que não satisfaz a equação original |
| Equação mista | equação que combina termos exponenciais e logarítmicos |
| Resolução numérica | encontrar a raiz de uma equação por aproximação sucessiva (ex.: `scipy.optimize.brentq`), quando não há solução algébrica simples |

## Conexões

**Isso depende de:**
- [`0204a_funcao_logaritmica.ipynb`](0204a_funcao_logaritmica.ipynb) (logaritmo como inversa da exponencial)
- [`0203a_logaritmos.ipynb`](0203a_logaritmos.ipynb) (propriedades operatórias e condições de existência)
- [`0202a_funcao_exponencial.ipynb`](0202a_funcao_exponencial.ipynb) (equações exponenciais simples, de base igual)

**Isso será usado em:**
- 📌 Inequações Exponenciais e Logarítmicas (mesma lógica geral, agora com desigualdades)
- 📌 Logaritmos Decimais (cálculos práticos com logaritmo decimal, característica e mantissa)
- [`10_matematica_financeira_comercial_e_estatistica_descritiva/`](../10_matematica_financeira_comercial_e_estatistica_descritiva/) (tempo para atingir um valor-alvo em juros compostos)